# 03. e + Ba⁺: сечения и константы скоростей для 455.403 и 493.408 нм

* A и ветвления — NIST ASD (известны все ветви: 6p ²P₃/₂ → 6s, 5d₅/₂, 5d₃/₂; 6p ²P₁/₂ → 6s,
  5d₃/₂; класс точности B) — `nb.BA_LINES`;
* сечения возбуждения — **считаются здесь** (готовых данных нет в наших источниках):
  модель `ba2` (`models.py`): один валентный электрон над [Xe]; 6s, 5d, 7s, 6d, 8s, 7d |
  6p, 7p, 4f, 5f (17 состояний), пороги — NIST.

Проверки модели: энергии против NIST; A против NIST; времена жизни 5d (E2) против
измеренных (впишите из литературы); связанные (N+1)-состояния = уровни Ba I.

Время (замер на 4 ядрах, J ≤ 3, 8 волн): самая большая матрица 10730, диагонализация одной
волны до 24 мин. Полный расчёт (J ≤ 20) — оценка: часы на 32 ядрах.

**Выход** (папка `ba2`): `sigma_ba2.xlsx` и `rates_ba2.xlsx` (все возбуждения между 17
состояниями, формат Ванга, номера NIST), `ba2_line_sigma.xlsx`, `ba2_k_line.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
JMAX = 20           # сходимость по J проверьте: 15 / 20 / 25
job = nb.Job(RUNS / "ba2", "ba2", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=30, de=0.005, te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

## 1. Энергии мишени

In [ ]:
W = RUNS / "ba2"
print((W / "target_table.txt").read_text())

## 2. A против NIST, времена жизни 5d

In [ ]:
NIST_A = {}
for x in nb.BA_LINES:
    if x["ion"] == "Ba II":
        NIST_A.update(x["A_branches"])
for r in nb.read_csv(W / "transitions_E1.csv"):
    wl = nb.fnum(r["wl_air_nm"])
    hit = [k for k in NIST_A if wl and abs(k - wl) < 0.05]
    if hit:
        a = float(r["A_exp"])
        print(f"{hit[0]:9.3f}  {r['upper_nist']:>14s} -> {r['lower_nist']:<14s}  DBSR {a:.3e}  NIST {NIST_A[hit[0]]:.3e}"
              f"  DBSR/NIST {a / NIST_A[hit[0]]:.2f}  vel/len {r['gauge_ratio']}")
MEASURED_TAU_S = {}        # впишите: {"5d 2D J=3/2": τ (с), "5d 2D J=5/2": τ (с)} со ссылкой
_, tot, _ = nb.branching_from_csv(W / "transitions_E2.csv")
for r in nb.read_csv(W / "transitions_E2.csv"):
    if r["upper_nist"].startswith("5d") and r["lower_nist"].startswith("6s"):
        u = int(r["upper_no"])
        print(f"E2 {r['upper_nist']}: τ = {1 / tot[u]:.1f} с  (изм.: {MEASURED_TAU_S.get(r['upper_nist'], '—')})")

## 3. Связанные состояния e + Ba⁺ = уровни Ba I (отдельный короткий прогон, J ≤ 4)

In [ ]:
jb = nb.Job(RUNS / "ba2_bound", "ba2", stage="bound", jmax=4, cores=CORES, mem=MEM, hd_threads=HD_THREADS)
jb.start()

In [ ]:
jb.status()
lv1 = db.nist.fetch_levels("Ba I")
for b in sorted(nb.read_csv(RUNS / "ba2_bound" / "bound_states.csv"), key=lambda b: float(b["E_cm"]))[:20]:
    e = float(b["E_cm"])                      # над самым нижним связанным состоянием (Ba I 6s2)
    near = min(lv1, key=lambda l: abs(l.energy_cm - e))
    print(f"{b['label']:>14s}  E = {e:9.1f} см-1   ближайший NIST: {near.config} {near.term} J={near.J:g}"
          f"  {near.energy_cm:9.1f}  Δ = {e - near.energy_cm:+7.0f}")

## 4. Сечения линий 455.4, 493.4 и константы скоростей

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz")
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states if s.name in cs.names}
name_of = {s.nist_label: n for n, s in st.items()}
TEa = np.array(TE)
sig, lab, k_line, lab_l = {}, {}, {}, {}
fig, ax = plt.subplots(figsize=(7, 4))
for x in [x for x in nb.BA_LINES if x["ion"] == "Ba II"]:
    u = name_of[x["upper"]]
    for i, si in st.items():
        if si.exp_energy_cm >= st[u].exp_energy_cm:
            continue
        E, s = cs.incident_energy(i), cs.sigma(i, u)
        m = np.isfinite(s) & (E > 0)
        key = (si.nist_no, st[u].nist_no, x["wl"])
        sig[key] = (E[m], x["branching"] * s[m])
        lab[key] = f"{si.nist_no}->{st[u].nist_no} ({x['wl']:.3f})"
        k_line[key] = x["branching"] * cs.rate(i, u, TEa * 11604.518)
        lab_l[key] = lab[key]
        if si.nist_no == 1:
            ax.plot(E[m], x["branching"] * s[m] / 1e-16, lw=0.8, label=f"{x['wl']} нм из 6s")
ax.set_xscale("log"); ax.set_xlabel("E, эВ"); ax.set_ylabel("σ линии, 10⁻¹⁶ см²"); ax.legend(); plt.show()
lv = [l for l in db.nist.fetch_levels("Ba II") if l.no in {s.nist_no for s in st.values()}]
reference.write_xlsx(W / "ba2_line_sigma.xlsx", lv, sig, sheet_of=lambda k: f"{k[2]:.3f}", labels=lab)
reference.write_rates_xlsx(W / "ba2_k_line.xlsx", TEa, k_line, lab_l,
                           note="BR * <sigma v> (cm3/s); sigma: pydbsr model ba2 (this run); BR: NIST ASD")
print("->", W / "ba2_line_sigma.xlsx", W / "ba2_k_line.xlsx")